In [1]:
# Standard libraries
import os

# Matplotlib
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# OpenCV
import cv2

## Constants

In [2]:
SOURCE_IMAGE_DIR = ".data/images"
GRAYSCALE_IMAGE_DIR = ".data/grayscale"

CATEGORY_LABELS = [
    "not_food",
    "italian_food",
    "japanese_food",
    "meat",
    "seafood",
    "soup",
    "salad",
    "dessert"
]

## Importing Grayscale Images.

In [ ]:
import numpy as np
import random

X_raw, y_raw = [], []

SAMPLES_PER_CATEGORY = 1024

for img_dir in CATEGORY_LABELS:
    sample_count = len(os.listdir(os.path.join(GRAYSCALE_IMAGE_DIR, img_dir)))
    print(f"Importing category: {img_dir} ({sample_count} samples)")

    samples = os.listdir(os.path.join(GRAYSCALE_IMAGE_DIR, img_dir))
    random.shuffle(samples)

    for filename in samples[:SAMPLES_PER_CATEGORY]:
        if not (filename.endswith(".jpg") or filename.endswith(".png")): continue

        img = cv2.imread(os.path.join(GRAYSCALE_IMAGE_DIR, img_dir, filename), cv2.IMREAD_GRAYSCALE)
        X_raw.append(img / 255.0)  # Normalize pixel values to [0, 1]
        y_raw.append(CATEGORY_LABELS.index(img_dir))

X = np.array(X_raw)
y = np.array(y_raw)

del X_raw, y_raw  # Free up memory

print("------------------------------")
print(f"Data imported. Total samples: {len(X)}")

Importing category: not_food (4319 samples)
Importing category: italian_food (3000 samples)
Importing category: japanese_food (3000 samples)
Importing category: meat (6000 samples)
Importing category: seafood (3000 samples)
Importing category: soup (2998 samples)
Importing category: salad (3000 samples)
Importing category: dessert (5000 samples)
------------------------------
Data imported. Total samples: 16384


In [4]:
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from cnn import ResNet, Bottleneck

net = ResNet(Bottleneck, [2, 2, 2, 2], num_classes=len(CATEGORY_LABELS), num_channels=1)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

X_tensor = torch.tensor(X, dtype=torch.float32).unsqueeze(1)  # Add channel dimension
y_tensor = torch.tensor(y, dtype=torch.int64)

dataset = TensorDataset(X_tensor, y_tensor)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

print(f"Data tensors created.\nX.shape = {X_tensor.shape}\ny.shape = {y_tensor.shape}")


: 

In [ ]:
len(dataloader)

In [ ]:
import time

for epoch in range(20):
    start = time.time()
    running_loss = 0.0
    for i, data in enumerate(dataloader, 0):
        inputs, labels = data
        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
    print(f"[Epoch {epoch + 1}] loss: {running_loss / len(dataloader):.3f} - Time: {time.time() - start:.2f} seconds")

    if epoch % 5 == 4:
        print("------------------------------")
        print(f"Saving model at epoch {epoch + 1}...")
        torch.save(net.state_dict(), f"resnet_food_classifier_epoch_{epoch + 1:02d}.pth")
        print("Model saved.")
        print("------------------------------")

print("Finished Training") 

In [ ]:
torch.save(net.state_dict(), "resnet_food_classifier.pth")

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.ensemble import RandomForestClassifier
import time

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model = RandomForestClassifier(n_estimators=100, random_state=42)

confusion_matrices = []
f1_scores = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    start = time.time()
    X_train, X_test = [X[i] for i in train_idx], [X[i] for i in test_idx]
    y_train, y_test = [y[i] for i in train_idx], [y[i] for i in test_idx]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    confusion_matrices.append(confusion_matrix(y_test, y_pred))
    f1_scores.append(f1_score(y_test, y_pred, average='macro'))
    print(classification_report(y_test, y_pred))
    print(f"Finished fold: {fold + 1} ({time.time() - start:.2f} seconds)")


print("Average F1 Score:", sum(f1_scores) / len(f1_scores))